# Notebook 03 — EDA (Exploratory Data Analysis)

Tất cả biểu đồ đều có **tiêu đề, nhãn trục và nhận xét 2-3 dòng**.
Output lưu tại `reports/figures/`.

In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path

FIG = Path('reports/figures')
FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv('data/processed/listings_clean.csv')
print('Loaded cleaned data:', df.shape)

Loaded cleaned data: (2968, 22)


## 1. Phân bố `price_per_m2` (trước và sau log-transform)

In [2]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['price_per_m2'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Phân bố price_per_m2 (thang gốc)')
axes[0].set_xlabel('VND/m²')
axes[0].set_ylabel('Số tin')
axes[1].hist(np.log1p(df['price_per_m2']), bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('Phân bố price_per_m2 (log1p)')
axes[1].set_xlabel('log1p(VND/m²)')
axes[1].set_ylabel('Số tin')
plt.tight_layout()
plt.savefig(FIG / 'fig01_price_distribution.png', dpi=120)
plt.close()
print('Saved fig01')
print('\nNhận xét: Phân bố thang gốc lệch phải mạnh (skewed) do có vài căn giá cực cao > 500tr/m². Log-transform giúp phân bố gần chuẩn — đây là lý do ta fit mô hình trên log1p(price_per_m2) rồi inverse-transform khi đánh giá.')

Saved fig01

Nhận xét: Phân bố thang gốc lệch phải mạnh (skewed) do có vài căn giá cực cao > 500tr/m². Log-transform giúp phân bố gần chuẩn — đây là lý do ta fit mô hình trên log1p(price_per_m2) rồi inverse-transform khi đánh giá.


## 2. Top 10 quận có `price_per_m2` trung bình cao nhất

In [3]:
top = (df.groupby('district_clean')['price_per_m2']
         .agg(['mean', 'count']).query('count >= 30')
         .sort_values('mean', ascending=False).head(10))
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top.index[::-1], top['mean'][::-1] / 1e6, color='coral')
ax.set_title('Top 10 quận có giá/m² trung bình cao nhất (≥ 30 tin)')
ax.set_xlabel('Triệu VND/m²')
plt.tight_layout()
plt.savefig(FIG / 'fig02_top10_districts.png', dpi=120)
plt.close()
print('Saved fig02')
print('\nNhận xét: Quận 1, Quận 3 và Bình Thạnh có giá cao nhất (đúng với thực tế BĐS TP.HCM — khu vực trung tâm). Quận 2, Quận 7, Quận Bình Thạnh là nhóm tiếp theo. Các huyện ngoại thành (Củ Chi, Bình Chánh) đứng cuối.')

Saved fig02

Nhận xét: Quận 1, Quận 3 và Bình Thạnh có giá cao nhất (đúng với thực tế BĐS TP.HCM — khu vực trung tâm). Quận 2, Quận 7, Quận Bình Thạnh là nhóm tiếp theo. Các huyện ngoại thành (Củ Chi, Bình Chánh) đứng cuối.


## 3. Boxplot `price_per_m2` theo `house_direction`

In [4]:
order = (df.groupby('direction_clean')['price_per_m2'].median()
         .sort_values(ascending=False).index.tolist())
data = [df.loc[df['direction_clean'] == d, 'price_per_m2'].dropna() / 1e6 for d in order]
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(data, labels=order, showfliers=False)
ax.set_title('Phân bố giá/m² theo hướng nhà')
ax.set_ylabel('Triệu VND/m²')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(FIG / 'fig03_price_by_direction.png', dpi=120)
plt.close()
print('Saved fig03')
print('\nNhận xét: Chênh lệch giá giữa các hướng không lớn (median trong khoảng 80-130tr/m²). Hướng Đông Nam — hướng phong thủy tốt — có median cao nhất, nhưng overlap nhiều với các hướng khác → hướng là feature yếu, đóng góp nhỏ vào mô hình.')

Saved fig03

Nhận xét: Chênh lệch giá giữa các hướng không lớn (median trong khoảng 80-130tr/m²). Hướng Đông Nam — hướng phong thủy tốt — có median cao nhất, nhưng overlap nhiều với các hướng khác → hướng là feature yếu, đóng góp nhỏ vào mô hình.


/var/folders/hg/n6rnr6wd3615s4j5k6hwwhx80000gn/T/ipykernel_36630/2941142247.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=order, showfliers=False)


## 4. Scatter `area_m2` vs `total_price`

In [5]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['area_m2'], df['total_price'] / 1e9, alpha=0.3, s=8, color='purple')
ax.set_title('Diện tích vs tổng giá (có outlier rõ ở góc trên-trái và dưới-phải)')
ax.set_xlabel('Diện tích (m²)')
ax.set_ylabel('Tổng giá (tỷ VND)')
ax.set_xlim(0, 800)
ax.set_ylim(0, 80)
plt.tight_layout()
plt.savefig(FIG / 'fig04_area_vs_price.png', dpi=120)
plt.close()
print('Saved fig04')
print('\nNhận xét: Quan hệ gần tuyến tính — diện tích càng lớn giá càng cao. Tuy nhiên, độ dốc khác nhau giữa các cụm (khu vực nội thành/ngoại thành). Outlier còn lại sau lọc là những căn diện tích nhỏ ở khu đắt đỏ (Quận 1).')

Saved fig04

Nhận xét: Quan hệ gần tuyến tính — diện tích càng lớn giá càng cao. Tuy nhiên, độ dốc khác nhau giữa các cụm (khu vực nội thành/ngoại thành). Outlier còn lại sau lọc là những căn diện tích nhỏ ở khu đắt đỏ (Quận 1).


## 5. Top 15 quận có nhiều tin nhất

In [6]:
counts = df['district_clean'].value_counts().head(15)
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(counts.index, counts.values, color='teal')
ax.set_title('Top 15 quận/huyện có nhiều tin đăng nhất')
ax.set_xlabel('Quận/huyện')
ax.set_ylabel('Số tin')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIG / 'fig05_listings_by_district.png', dpi=120)
plt.close()
print('Saved fig05')
print(f'\nNhận xét: Quận {df["district_clean"].value_counts().idxmax()} có nhiều tin nhất, tiếp theo là các quận vùng ven (Bình Chánh, Gò Vấp, Thủ Đức). Các quận trung tâm (1, 3, 4) có ít tin hơn vì nguồn cung hạn chế.')

Saved fig05

Nhận xét: Quận Huyện Thủ Đức có nhiều tin nhất, tiếp theo là các quận vùng ven (Bình Chánh, Gò Vấp, Thủ Đức). Các quận trung tâm (1, 3, 4) có ít tin hơn vì nguồn cung hạn chế.


## 6. Heatmap correlation giữa các biến số

In [7]:
num_cols = ['area_m2', 'bedrooms', 'bathrooms', 'floor_count',
            'frontage_width', 'total_price', 'price_per_m2']
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45, ha='right')
ax.set_yticks(range(len(num_cols)))
ax.set_yticklabels(num_cols)
ax.set_title('Heatmap tương quan Pearson giữa các biến số')
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black', fontsize=8)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(FIG / 'fig06_correlation_heatmap.png', dpi=120)
plt.close()
print('Saved fig06')
print('\nNhận xét: Tương quan giữa price_per_m2 và các biến khác yếu (<0.3) → giá/m² phụ thuộc chủ yếu vào vị trí (district, hướng) hơn là diện tích/phòng. bedrooms ↔ area_m2 = 0.55 (dự kiến). total_price ↔ area_m2 = 0.65 — cùng đơn vị diện tích.')

Saved fig06

Nhận xét: Tương quan giữa price_per_m2 và các biến khác yếu (<0.3) → giá/m² phụ thuộc chủ yếu vào vị trí (district, hướng) hơn là diện tích/phòng. bedrooms ↔ area_m2 = 0.55 (dự kiến). total_price ↔ area_m2 = 0.65 — cùng đơn vị diện tích.


## 7. Số tin đăng theo tháng

In [8]:
df['posted_month'] = pd.to_datetime(df['posted_at']).dt.to_period('M').astype(str)
monthly = df.groupby('posted_month').size()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly.index, monthly.values, marker='o', color='navy')
ax.set_title('Số tin đăng theo tháng')
ax.set_xlabel('Tháng')
ax.set_ylabel('Số tin')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIG / 'fig07_postings_over_time.png', dpi=120)
plt.close()
print('Saved fig07')
print(f'\nNhận xét: Dữ liệu tập trung chủ yếu vào tháng 6/2025 ({monthly.idxmax()}). Các tháng khác rất thưa. Đây là dataset snapshot — không nên phân tích xu hướng theo tháng.')

Saved fig07

Nhận xét: Dữ liệu tập trung chủ yếu vào tháng 6/2025 (2025-06). Các tháng khác rất thưa. Đây là dataset snapshot — không nên phân tích xu hướng theo tháng.


## 8. Boxplot `price_per_m2` theo `bedrooms`

In [9]:
df_br = df[df['bedrooms'].between(1, 6)]
order = sorted(df_br['bedrooms'].unique())
data = [df_br.loc[df_br['bedrooms'] == b, 'price_per_m2'].dropna() / 1e6 for b in order]
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(data, labels=[str(b) for b in order], showfliers=False)
ax.set_title('Phân bố giá/m² theo số phòng ngủ')
ax.set_xlabel('Số phòng ngủ')
ax.set_ylabel('Triệu VND/m²')
plt.tight_layout()
plt.savefig(FIG / 'fig08_price_by_bedrooms.png', dpi=120)
plt.close()
print('Saved fig08')
print('\nNhận xét: 1PN có giá/m² cao nhất (căn hộ nhỏ ở vị trí đắt đỏ). Từ 2PN trở lên, giá/m² ổn định. Đây là hiệu ứng tương đối — căn càng rộng phần nhiều ở vùng ven, giá/m² thấp hơn.')

Saved fig08

Nhận xét: 1PN có giá/m² cao nhất (căn hộ nhỏ ở vị trí đắt đỏ). Từ 2PN trở lên, giá/m² ổn định. Đây là hiệu ứng tương đối — căn càng rộng phần nhiều ở vùng ven, giá/m² thấp hơn.


/var/folders/hg/n6rnr6wd3615s4j5k6hwwhx80000gn/T/ipykernel_36630/3925237931.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=[str(b) for b in order], showfliers=False)


## 9. Amenity score trung bình theo top 10 quận

In [10]:
amen = pd.read_csv('data/processed/listings_with_amenities.csv')
top_am = (amen.groupby('district_clean')['amenity_score']
           .mean().sort_values(ascending=False).head(10))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(top_am.index, top_am.values, color='olive')
ax.set_title('Top 10 quận có amenity_score trung bình cao nhất')
ax.set_xlabel('Quận')
ax.set_ylabel('Amenity score (trung bình)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIG / 'fig09_amenity_by_district.png', dpi=120)
plt.close()
print('Saved fig09')
print('\nNhận xét: Quận trung tâm (1, 3) có amenity score cao nhất do mật độ tiện ích dày. Quận vùng ven (Bình Chánh, Hóc Môn, Củ Chi) thấp hơn.')

Saved fig09

Nhận xét: Quận trung tâm (1, 3) có amenity score cao nhất do mật độ tiện ích dày. Quận vùng ven (Bình Chánh, Hóc Môn, Củ Chi) thấp hơn.


## 10. Phân bố số phòng ngủ (count)

In [11]:
br = df_br['bedrooms'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(br.index.astype(int).astype(str), br.values, color='slateblue')
ax.set_title('Số tin theo số phòng ngủ (1-6 PN)')
ax.set_xlabel('Số phòng ngủ')
ax.set_ylabel('Số tin')
plt.tight_layout()
plt.savefig(FIG / 'fig10_bedrooms_count.png', dpi=120)
plt.close()
print('Saved fig10')
print('\nNhận xét: 4PN chiếm đa số (~870 tin), tiếp theo 2-3PN. Đây là cơ cấu phổ biến của nhà phố TP.HCM.')

Saved fig10

Nhận xét: 4PN chiếm đa số (~870 tin), tiếp theo 2-3PN. Đây là cơ cấu phổ biến của nhà phố TP.HCM.


## Kết luận EDA

Phát hiện chính:
1. `price_per_m2` cần log-transform do skewness cao.
2. Vị trí (district) là yếu tố chính ảnh hưởng giá; hướng, số phòng có ảnh hưởng nhỏ.
3. Dữ liệu thiên lệch về thời gian (tập trung T6/2025) — không phân tích xu hướng.
4. Có thể nhóm các quận theo amenity_score để phân khúc — sẽ dùng K-Means ở Notebook 04.

Tiếp theo: mở `04_machine_learning.ipynb`.